MOBILE LEGENDS

testing road to mythical glory

# Google Play Store ML Activity

End-to-end workflow: inspect, clean, visualize, model, evaluate, and interpret results.

AI SHIT GO BRRR

In [1]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sns.set_theme(style='whitegrid')

## 1) Load Dataset

In [ ]:
df_raw = pd.read_csv('dataset/googleplaystore.csv')
df_raw.head()

## 2) Inspect Data

In [ ]:
print('Shape:', df_raw.shape)
display(df_raw.info())
display(df_raw.describe(include='all').T.head(15))
display(df_raw.isna().sum().sort_values(ascending=False).head(10))

## 3) Clean Data

Main cleaning steps:
- Remove duplicate apps
- Convert `Reviews`, `Installs`, `Price`, and `Size` to numeric
- Remove invalid ratings (`Rating > 5`)
- Fill remaining missing values for key fields

In [ ]:
df = df_raw.copy()

# Keep one row per app
df = df.drop_duplicates(subset='App').reset_index(drop=True)

# Rating cleanup
df.loc[df['Rating'] > 5, 'Rating'] = np.nan
df['Rating'] = df['Rating'].fillna(df['Rating'].median())

# Reviews to numeric
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')
df['Reviews'] = df['Reviews'].fillna(df['Reviews'].median())

# Installs like 1,000+ -> 1000
df['Installs'] = (
    df['Installs']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
)
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')
df['Installs'] = df['Installs'].fillna(df['Installs'].median())

# Price like $4.99 -> 4.99
df['Price'] = df['Price'].astype(str).str.replace('$', '', regex=False)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce').fillna(0)

# Size like 14M / 300k -> MB
def size_to_mb(x):
    s = str(x).strip()
    if s == 'Varies with device' or s == 'nan':
        return np.nan
    if s.endswith('M'):
        return float(s[:-1])
    if s.endswith('k'):
        return float(s[:-1]) / 1024
    return np.nan

df['SizeMB'] = df['Size'].apply(size_to_mb)
df['SizeMB'] = df['SizeMB'].fillna(df['SizeMB'].median())

# Keep needed columns for this task
model_df = df[['Category', 'Content Rating', 'Genres', 'Rating', 'Reviews', 'Installs', 'Price', 'SizeMB', 'Type']].copy()
model_df = model_df.dropna(subset=['Type']).reset_index(drop=True)

print('Cleaned shape:', model_df.shape)
model_df.head()

In [ ]:
print('Remaining missing values:')
model_df.isna().sum().sort_values(ascending=False)

## 4) Exploratory Visualizations

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(model_df['Rating'], bins=20, kde=True)
plt.title('Distribution of App Ratings')
plt.xlabel('Rating')
plt.show()

In [ ]:
top_categories = model_df['Category'].value_counts().head(10).index
plot_df = model_df[model_df['Category'].isin(top_categories)]

plt.figure(figsize=(10, 5))
sns.boxplot(data=plot_df, x='Category', y='Rating')
plt.title('Ratings by Top 10 Categories')
plt.xticks(rotation=45, ha='right')
plt.show()

## 5) Prepare Features and Target

Task: classification where target is `Type` (`Free` vs `Paid`).

In [ ]:
data = model_df.copy()
data['Type'] = data['Type'].map({'Free': 0, 'Paid': 1})

# One-hot encode categorical columns
X = pd.get_dummies(data.drop(columns=['Type']), columns=['Category', 'Content Rating', 'Genres'], drop_first=True)
y = data['Type']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('X_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)
print('Class balance in y (0=Free, 1=Paid):')
print(y.value_counts(normalize=True))

## 6) Train Models

In [ ]:
# Scale for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=2000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
pred_lr = log_reg.predict(X_test_scaled)

tree = DecisionTreeClassifier(max_depth=8, random_state=42)
tree.fit(X_train, y_train)
pred_tree = tree.predict(X_test)

## 7) Evaluate Models

In [ ]:
acc_lr = accuracy_score(y_test, pred_lr)
acc_tree = accuracy_score(y_test, pred_tree)

print(f'Logistic Regression Accuracy: {acc_lr:.4f}')
print(f'Decision Tree Accuracy      : {acc_tree:.4f}')

print('\nClassification Report - Logistic Regression')
print(classification_report(y_test, pred_lr))

print('Classification Report - Decision Tree')
print(classification_report(y_test, pred_tree))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm_lr = confusion_matrix(y_test, pred_lr)
cm_tree = confusion_matrix(y_test, pred_tree)

sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Logistic Regression Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_tree, annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('Decision Tree Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 8) Interpretation Guide

Write your final insights here:
1. Which model performed better and by how much?
2. Which features seem important for distinguishing Free vs Paid apps?
3. What data quality issues likely affected results?
4. What would you improve next (more features, balancing, other models)?